Step 1: Dependencies and Imports

In [3]:
# Install any missing dependencies (Colab usually has pandas/sklearn pre-installed, but let's be safe)
!pip install joblib scikit-learn pandas numpy --quiet

import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from google.colab import files  # Special module to download the file directly

Step 2: Data Processing and Engineering Functions

In [4]:
def download_and_process_nasa_data():
    print("🚀 Fetching authentic NASA Turbofan (C-MAPSS) Dataset...")
    url = "https://raw.githubusercontent.com/mapr-demos/predictive-maintenance/master/notebooks/jupyter/Dataset/CMAPSSData/train_FD001.txt"

    index_names = ['unit_nr', 'time_cycles']
    setting_names = ['setting_1', 'setting_2', 'setting_3']
    sensor_names = [f's_{i}' for i in range(1, 22)]
    col_names = index_names + setting_names + sensor_names

    # Read the space-separated NASA telemetry file
    df = pd.read_csv(url, sep=r"\s+", header=None, names=col_names)
    print(f"✅ Loaded {len(df)} operational engine snapshots directly.")

    # --- PROGNOSTICS FEATURE ENGINEERING: CALCULATE RUL ---
    max_cycles = df.groupby('unit_nr')['time_cycles'].max().reset_index()
    max_cycles.columns = ['unit_nr', 'max_cycles']
    df = df.merge(max_cycles, on='unit_nr', how='left')

    # Remaining Useful Life (RUL)
    df['RUL'] = df['max_cycles'] - df['time_cycles']

    # Binary target classification: Is failure imminent within 30 cycles?
    df['failure_target'] = np.where(df['RUL'] <= 30, 1, 0)

    # Select high-importance physical sensors
    target_sensors = ['s_2', 's_3', 's_4', 's_11']

    # Smooth out raw instrumentation noise using a 5-cycle rolling window average
    for feature in target_sensors:
        df[f'{feature}_smoothed'] = df.groupby('unit_nr')[feature].transform(
            lambda x: x.rolling(window=5, min_periods=1).mean()
        )

    return df

Step 3: Model Execution and Automated Export

In [5]:
# 1. Load and engineer features
df = download_and_process_nasa_data()

# 2. Isolate features and targets
features = ['s_2_smoothed', 's_3_smoothed', 's_4_smoothed', 's_11_smoothed']
X = df[features]
y = df['failure_target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("\n🧠 Training Random Forest Operational Safety Model...")
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

print("\n📋 NASA Fleet Model Evaluation Ledger:")
print(classification_report(y_test, model.predict(X_test)))

# 3. Save the binary model to the local Colab workspace
model_filename = "infrastructure_model.joblib"
joblib.dump(model, model_filename)
print(f"\n📦 Saved model state cleanly to workspace: {model_filename}")

# 4. Trigger browser download dialog
print("📥 Initializing automatic download to your computer...")
files.download(model_filename)

🚀 Fetching authentic NASA Turbofan (C-MAPSS) Dataset...
✅ Loaded 20631 operational engine snapshots directly.

🧠 Training Random Forest Operational Safety Model...

📋 NASA Fleet Model Evaluation Ledger:
              precision    recall  f1-score   support

           0       0.98      0.95      0.97      3507
           1       0.77      0.90      0.83       620

    accuracy                           0.94      4127
   macro avg       0.87      0.93      0.90      4127
weighted avg       0.95      0.94      0.95      4127


📦 Saved model state cleanly to workspace: infrastructure_model.joblib
📥 Initializing automatic download to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>